# NewsCLIPpings VisualNews Mapping

Inspect caption and image references in memory. Raw files stay unchanged; no assets or mapping files are created.

## Setup

In [1]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd

working_directory = Path.cwd()
project_root = working_directory
if working_directory.name == "notebooks":
    project_root = working_directory.parent

newsclippings_directory = project_root / "data" / "raw" / "newsclippings"
metadata_directory = project_root / "data" / "raw" / "visualnews_metadata"
splits = ["train", "val", "test"]

## Load Data

In [2]:
newsclippings = {}
visualnews_metadata = {}
file_rows = []

for split in splits:
    news_path = newsclippings_directory / f"{split}.json"
    metadata_path = metadata_directory / f"{split}.json"

    with news_path.open("r", encoding="utf-8") as handle:
        news_object = json.load(handle)
    with metadata_path.open("r", encoding="utf-8") as handle:
        metadata_object = json.load(handle)

    newsclippings[split] = news_object["annotations"]
    visualnews_metadata[split] = metadata_object
    file_rows.append({
        "split": split,
        "newsclippings_rows": len(newsclippings[split]),
        "visualnews_records": len(metadata_object),
    })

display(pd.DataFrame(file_rows))

,split,newsclippings_rows,visualnews_records
0,train,71072,385003
1,val,7024,40830
2,test,7264,40915


## Build Lookup

Combine the metadata splits only after checking that record IDs are globally unique.

In [3]:
metadata_id_counts = Counter()
key_id_mismatches = 0

for split in splits:
    metadata_records = visualnews_metadata[split]
    for key, record in metadata_records.items():
        metadata_id = record["id"]
        if type(metadata_id) is not int or metadata_id < 0:
            raise ValueError(f"{split}: invalid metadata ID {metadata_id!r}")

        metadata_id_counts[metadata_id] += 1
        if str(metadata_id) != key:
            key_id_mismatches += 1

repeated_metadata_ids = []
duplicate_id_rows = 0
for metadata_id, count in metadata_id_counts.items():
    if count > 1:
        repeated_metadata_ids.append({"id": metadata_id, "record_count": count})
        duplicate_id_rows += count - 1

lookup_check = pd.DataFrame([{
    "metadata_records": sum(metadata_id_counts.values()),
    "unique_ids": len(metadata_id_counts),
    "duplicated_ids": len(repeated_metadata_ids),
    "duplicate_id_rows": duplicate_id_rows,
    "key_id_mismatches": key_id_mismatches,
}])
display(lookup_check)

,metadata_records,unique_ids,duplicated_ids,duplicate_id_rows,key_id_mismatches
0,466748,466748,0,0,0


In [4]:
if repeated_metadata_ids:
    display(pd.DataFrame(repeated_metadata_ids).head(10))
    raise ValueError("Duplicate metadata IDs make the combined lookup ambiguous. Stop here.")
if key_id_mismatches:
    raise ValueError("Metadata keys disagree with record IDs. Review before mapping.")

visualnews_by_id = {}
metadata_split_by_id = {}
for split in splits:
    for record in visualnews_metadata[split].values():
        metadata_id = record["id"]
        visualnews_by_id[metadata_id] = record
        metadata_split_by_id[metadata_id] = split

print("Combined lookup records:", f"{len(visualnews_by_id):,}")

Combined lookup records: 466,748


`id` supplies the submitted caption; `image_id` supplies the query image and its original caption.

In [5]:
mapped_rows = []

for split in splits:
    for annotation in newsclippings[split]:
        caption_id = annotation["id"]
        image_id = annotation["image_id"]
        caption_record = visualnews_by_id.get(caption_id)
        image_record = visualnews_by_id.get(image_id)

        caption_found = caption_record is not None
        image_found = image_record is not None
        both_found = caption_found and image_found
        same_record = both_found and caption_record is image_record

        caption_fields = caption_record if caption_found else {}
        image_fields = image_record if image_found else {}
        captions_equal = None
        if both_found:
            captions_equal = caption_fields.get("caption") == image_fields.get("caption")

        mapped_rows.append({
            "split": split, "id": caption_id, "image_id": image_id,
            "falsified": annotation["falsified"],
            "submitted_caption": caption_fields.get("caption"),
            "submitted_caption_source": caption_fields.get("source"),
            "submitted_caption_image_path": caption_fields.get("image_path"),
            "submitted_caption_article_path": caption_fields.get("article_path"),
            "submitted_caption_full_article_path": caption_fields.get("full_article_path"),
            "query_image_original_caption": image_fields.get("caption"),
            "query_image_path": image_fields.get("image_path"),
            "query_image_source": image_fields.get("source"),
            "query_image_article_path": image_fields.get("article_path"),
            "query_image_full_article_path": image_fields.get("full_article_path"),
            "caption_timestamp": caption_fields.get("timestamp"),
            "image_timestamp": image_fields.get("timestamp"),
            "caption_metadata_split": metadata_split_by_id.get(caption_id),
            "image_metadata_split": metadata_split_by_id.get(image_id),
            "caption_found": caption_found, "image_found": image_found,
            "both_found": both_found, "same_metadata_record": same_record,
            "captions_equal": captions_equal,
        })

mapped_table = pd.DataFrame(mapped_rows)
print("In-memory annotation rows:", f"{len(mapped_table):,}")

In-memory annotation rows: 85,360


## Mapping Coverage

In [6]:
coverage_rows = []
for split in splits:
    split_rows = mapped_table[mapped_table["split"] == split]
    total_rows = len(split_rows)
    caption_found = int(split_rows["caption_found"].sum())
    image_found = int(split_rows["image_found"].sum())
    both_found = int(split_rows["both_found"].sum())

    coverage_rows.append({
        "split": split, "total_rows": total_rows,
        "id_found": caption_found, "id_missing": total_rows - caption_found,
        "image_id_found": image_found, "image_id_missing": total_rows - image_found,
        "both_resolve": both_found, "either_fails": total_rows - both_found,
    })
coverage_table = pd.DataFrame(coverage_rows)
display(coverage_table)
print("Overall both-ID coverage:", f"{mapped_table['both_found'].mean():.2%}")

,split,total_rows,id_found,id_missing,image_id_found,image_id_missing,both_resolve,either_fails
0,train,71072,71072,0,71072,0,71072,0
1,val,7024,7024,0,7024,0,7024,0
2,test,7264,7264,0,7264,0,7264,0


Overall both-ID coverage: 100.00%


In [7]:
coverage_percentages = coverage_table[["split"]].copy()
coverage_columns = [
    "id_found", "id_missing", "image_id_found", "image_id_missing",
    "both_resolve", "either_fails",
]
for column in coverage_columns:
    percentages = 100 * coverage_table[column] / coverage_table["total_rows"]
    coverage_percentages[f"{column}_pct"] = percentages

display(coverage_percentages)

,split,id_found_pct,id_missing_pct,image_id_found_pct,image_id_missing_pct,both_resolve_pct,either_fails_pct
0,train,100.0,0.0,100.0,0.0,100.0,0.0
1,val,100.0,0.0,100.0,0.0,100.0,0.0
2,test,100.0,0.0,100.0,0.0,100.0,0.0


In [8]:
mapping_split_rows = []
for split in splits:
    split_rows = mapped_table[mapped_table["split"] == split]
    for role, column, resolved_column in (
        ("caption", "caption_metadata_split", "caption_found"),
        ("image", "image_metadata_split", "image_found"),
    ):
        resolved_rows = split_rows[split_rows[resolved_column]]
        same_split = resolved_rows[column] == split
        mapping_split_rows.append({
            "newsclippings_split": split, "role": role,
            "same_metadata_split": int(same_split.sum()),
            "other_metadata_split": int((~same_split).sum()),
        })
display(pd.DataFrame(mapping_split_rows))

,newsclippings_split,role,same_metadata_split,other_metadata_split
0,train,caption,71072,0
1,train,image,71072,0
2,val,caption,7024,0
3,val,image,7024,0
4,test,caption,7264,0
5,test,image,7264,0


## Pristine Examples

In [9]:
resolved_rows = mapped_table[mapped_table["both_found"]]
pristine_rows = resolved_rows[resolved_rows["falsified"] == False]
falsified_rows = resolved_rows[resolved_rows["falsified"] == True]

pristine_check_rows = []
for split in splits:
    split_rows = pristine_rows[pristine_rows["split"] == split]
    equal_ids = split_rows["id"] == split_rows["image_id"]
    pristine_check_rows.append({
        "split": split, "resolved_pristine_rows": len(split_rows),
        "equal_ids": int(equal_ids.sum()),
        "equal_captions": int(split_rows["captions_equal"].sum()),
        "same_metadata_record": int(split_rows["same_metadata_record"].sum()),
    })
display(pd.DataFrame(pristine_check_rows))

,split,resolved_pristine_rows,equal_ids,equal_captions,same_metadata_record
0,train,35536,35536,35536,35536
1,val,3512,3512,3512,3512
2,test,3632,3632,3632,3632


In [10]:
pristine_examples = pristine_rows.sample(n=min(8, len(pristine_rows)), random_state=42)
pristine_columns = [
    "split", "id", "image_id", "submitted_caption",
    "query_image_original_caption", "query_image_path", "query_image_source",
]
with pd.option_context("display.max_colwidth", None):
    display(pristine_examples[pristine_columns])

,split,id,image_id,submitted_caption,query_image_original_caption,query_image_path,query_image_source
34282,train,113591,113591,President Obama shakes hands during his 2012 State of the Union address AP photo,President Obama shakes hands during his 2012 State of the Union address AP photo,visual_news/origin/washington_post/images/0425/996.jpg,washington_post
78056,val,307320,307320,Game 5 in Houston Rockets 103 Mavericks 94 Houston guard James Harden gets fired up after a crucial fourthquarter bucket,Game 5 in Houston Rockets 103 Mavericks 94 Houston guard James Harden gets fired up after a crucial fourthquarter bucket,visual_news/origin/usa_today/images/0714/911.jpg,usa_today
20730,train,369682,369682,Duchess Kate and Prince William visited Steward s Academy on Sept 16 2016 in Harlow England as part of their Heads Together campaign to promote children s mental health,Duchess Kate and Prince William visited Steward s Academy on Sept 16 2016 in Harlow England as part of their Heads Together campaign to promote children s mental health,visual_news/origin/usa_today/images/0605/169.jpg,usa_today
32500,train,416745,416745,Tiger Woods places his ball before a birdie attempt on the 15th green during the second round of the Bridgestone Invitational golf tournament,Tiger Woods places his ball before a birdie attempt on the 15th green during the second round of the Bridgestone Invitational golf tournament,visual_news/origin/usa_today/images/0027/999.jpg,usa_today
28558,train,1532173,1532173,Lachlan Rupert and James Murdoch,Lachlan Rupert and James Murdoch,visual_news/origin/bbc/images/0158/683.jpg,bbc
29234,train,681124,681124,Michael Bryant left and Richard Pasco in Murmuring Judges by David Hare National Theatre 1991,Michael Bryant left and Richard Pasco in Murmuring Judges by David Hare National Theatre 1991,visual_news/origin/guardian/images/0384/987.jpg,guardian
5964,train,140971,140971,A rocket carrying a US Navy satellite launches from Cape Canaveral Fla Russian hackers are using older commercial satellites to siphon off data and cover their tracks,A rocket carrying a US Navy satellite launches from Cape Canaveral Fla Russian hackers are using older commercial satellites to siphon off data and cover their tracks,visual_news/origin/washington_post/images/0141/940.jpg,washington_post
41562,train,1176510,1176510,Ommmm new hope compassion meaning for Don Draper in the final Mad Men Photograph courtesy of AMC,Ommmm new hope compassion meaning for Don Draper in the final Mad Men Photograph courtesy of AMC,visual_news/origin/guardian/images/0791/047.jpg,guardian


## Falsified Examples

In [11]:
falsified_check_rows = []
for split in splits:
    split_rows = falsified_rows[falsified_rows["split"] == split]
    different_ids = split_rows["id"] != split_rows["image_id"]
    falsified_check_rows.append({
        "split": split, "resolved_falsified_rows": len(split_rows),
        "different_ids": int(different_ids.sum()),
        "same_metadata_record": int(split_rows["same_metadata_record"].sum()),
    })
display(pd.DataFrame(falsified_check_rows))

,split,resolved_falsified_rows,different_ids,same_metadata_record
0,train,35536,35536,0
1,val,3512,3512,0
2,test,3632,3632,0


In [12]:
example_columns = [
    "split", "falsified", "id", "image_id", "submitted_caption",
    "query_image_original_caption", "query_image_path",
    "submitted_caption_source", "query_image_source",
]
falsified_examples = falsified_rows.sample(n=min(12, len(falsified_rows)), random_state=42)
with pd.option_context("display.max_colwidth", None):
    display(falsified_examples[example_columns])

,split,falsified,id,image_id,submitted_caption,query_image_original_caption,query_image_path,submitted_caption_source,query_image_source
34283,train,True,113591,39678,President Obama shakes hands during his 2012 State of the Union address AP photo,President Obama pauses while speaking at the Federal Trade Commission offices at the Constitution Center,visual_news/origin/washington_post/images/0356/181.jpg,washington_post,washington_post
78057,val,True,307320,209234,Game 5 in Houston Rockets 103 Mavericks 94 Houston guard James Harden gets fired up after a crucial fourthquarter bucket,Game 4 in Oklahoma City Thunder 111 Spurs 97 Thunder forward Kevin Durant gets fired up after a big fourthquarter bucket,visual_news/origin/usa_today/images/0595/625.jpg,usa_today,usa_today
20731,train,True,369682,344086,Duchess Kate and Prince William visited Steward s Academy on Sept 16 2016 in Harlow England as part of their Heads Together campaign to promote children s mental health,Britain s Prince William and Kate Duchess of Cambridge and the Prince George during a visit to the Sensational Butterflies exhibition at the Natural History Museum London,visual_news/origin/usa_today/images/0352/152.jpg,usa_today,usa_today
32501,train,True,416745,565837,Tiger Woods places his ball before a birdie attempt on the 15th green during the second round of the Bridgestone Invitational golf tournament,Tiger Woods plays from the bunker at the 18th at the US Open at the Olympic Club,visual_news/origin/guardian/images/0274/629.jpg,usa_today,guardian
28559,train,True,1532173,705991,Lachlan Rupert and James Murdoch,James Murdoch called Vince Cable on day of the BSkyB bid,visual_news/origin/guardian/images/0522/464.jpg,bbc,guardian
29235,train,True,681124,1127505,Michael Bryant left and Richard Pasco in Murmuring Judges by David Hare National Theatre 1991,Hare photographed backstage at the National Theatre in February 1985 the year of his brilliant satire on the Murdoch press Pravda,visual_news/origin/guardian/images/0728/763.jpg,guardian,guardian
5965,train,True,140971,335348,A rocket carrying a US Navy satellite launches from Cape Canaveral Fla Russian hackers are using older commercial satellites to siphon off data and cover their tracks,The Soyuz TMA15M rocket launches from the Baikonur Cosmodrome in Kazakhstan on 24 Nov 24,visual_news/origin/usa_today/images/0337/630.jpg,washington_post,usa_today
41563,train,True,1176510,1142221,Ommmm new hope compassion meaning for Don Draper in the final Mad Men Photograph courtesy of AMC,Anyway none of this matters Kelly Reilly as Kate and Clive Owen as Deeley in Old Times,visual_news/origin/guardian/images/0751/207.jpg,guardian,guardian
82979,test,True,624023,1464135,Urs Rohner CEO of Credit Suisse participates in a panel session in Bern Switzerland on Tuesday,Chris Huhne is among the ministers expected to address delegates at next week s Lib Dem conference,visual_news/origin/bbc/images/0089/464.jpg,guardian,bbc
49251,train,True,478730,153697,Alina Cojocaru and Alejandro Virelles in Spring and Fall,Russia s Nikita Katsalapov and Russia s Elena Ilinykh perform in the Figure Skating Ice Dance Free Dance,visual_news/origin/washington_post/images/0069/607.jpg,guardian,washington_post


## Caption Relationships

Exact string comparisons only; no semantic similarity judgments.

In [13]:
caption_relationship_rows = []
for split in splits:
    split_rows = resolved_rows[resolved_rows["split"] == split]
    for falsified in (False, True):
        class_rows = split_rows[split_rows["falsified"] == falsified]
        exact_matches = int(class_rows["captions_equal"].sum())
        caption_relationship_rows.append({
            "split": split, "falsified": falsified,
            "resolved_rows": len(class_rows), "exact_caption_matches": exact_matches,
            "different_captions": len(class_rows) - exact_matches,
        })
display(pd.DataFrame(caption_relationship_rows))

,split,falsified,resolved_rows,exact_caption_matches,different_captions
0,train,False,35536,35536,0
1,train,True,35536,0,35536
2,val,False,3512,3512,0
3,val,True,3512,0,3512
4,test,False,3632,3632,0
5,test,True,3632,0,3632


## Source Relationships

Publisher differences are not evidence of misinformation.

In [14]:
source_relationship_rows = []
for split in splits:
    split_rows = falsified_rows[falsified_rows["split"] == split]
    same_source = split_rows["submitted_caption_source"] == split_rows["query_image_source"]
    source_relationship_rows.append({
        "split": split, "falsified_rows": len(split_rows),
        "same_source": int(same_source.sum()),
        "different_source": int((~same_source).sum()),
    })
source_relationship_table = pd.DataFrame(source_relationship_rows)
display(source_relationship_table)

source_pair_columns = ["submitted_caption_source", "query_image_source"]
source_pair_counts = falsified_rows.groupby(source_pair_columns).size()
source_pair_table = source_pair_counts.reset_index(name="count")
source_pair_table = source_pair_table.sort_values("count", ascending=False)
display(source_pair_table)

,split,falsified_rows,same_source,different_source
0,train,35536,16671,18865
1,val,3512,1649,1863
2,test,3632,1662,1970


,submitted_caption_source,query_image_source,count
5,guardian,guardian,11567
10,usa_today,usa_today,4465
1,bbc,guardian,3009
6,guardian,usa_today,2951
9,usa_today,guardian,2608
4,guardian,bbc,2558
15,washington_post,washington_post,2401
7,guardian,washington_post,2339
14,washington_post,usa_today,2229
11,usa_today,washington_post,2198


## Path Checks

Check populated metadata references on both sides. Physical assets are not checked.

In [15]:
path_columns = {
    "caption": ["submitted_caption_image_path", "submitted_caption_article_path", "submitted_caption_full_article_path"],
    "image": ["query_image_path", "query_image_article_path", "query_image_full_article_path"],
}
path_check_rows = []
for split in splits:
    split_rows = mapped_table[mapped_table["split"] == split]
    for role, columns in path_columns.items():
        for column in columns:
            populated_count = 0
            for value in split_rows[column]:
                if type(value) is str and value.strip():
                    populated_count += 1
            path_check_rows.append({
                "split": split, "role": role, "field": column,
                "populated": populated_count,
                "missing_or_blank": len(split_rows) - populated_count,
            })
path_check_table = pd.DataFrame(path_check_rows)
display(path_check_table)

,split,role,field,populated,missing_or_blank
0,train,caption,submitted_caption_image_path,71072,0
1,train,caption,submitted_caption_article_path,71072,0
2,train,caption,submitted_caption_full_article_path,71072,0
3,train,image,query_image_path,71072,0
4,train,image,query_image_article_path,71072,0
5,train,image,query_image_full_article_path,71072,0
6,val,caption,submitted_caption_image_path,7024,0
7,val,caption,submitted_caption_article_path,7024,0
8,val,caption,submitted_caption_full_article_path,7024,0
9,val,image,query_image_path,7024,0


## Missing IDs

In [16]:
missing_mapping_counts = Counter()
for split in splits:
    for annotation in newsclippings[split]:
        for role, field in (("caption", "id"), ("image", "image_id")):
            metadata_id = annotation[field]
            if metadata_id not in visualnews_by_id:
                missing_mapping_counts[(split, role, field, metadata_id)] += 1

missing_mapping_rows = []
for key, count in missing_mapping_counts.items():
    split, role, field, metadata_id = key
    missing_mapping_rows.append({
        "split": split, "role": role, "id_field": field,
        "missing_id": metadata_id, "count": count,
    })
missing_mapping_columns = ["split", "role", "id_field", "missing_id", "count"]
missing_mapping_table = pd.DataFrame(missing_mapping_rows, columns=missing_mapping_columns)
print("Missing mapping occurrences:", sum(missing_mapping_counts.values()))
display(missing_mapping_table)
print("All annotation rows retained:", len(mapped_table) == sum(len(rows) for rows in newsclippings.values()))

Missing mapping occurrences: 0


,split,role,id_field,missing_id,count


All annotation rows retained: True


## Manual Review

Seed 42; ten examples per class, drawn as four train, three validation, and three test rows. Review metadata roles only; images are not available.

In [17]:
review_counts = {"train": 4, "val": 3, "test": 3}
review_samples = []
for falsified in (False, True):
    for split, sample_count in review_counts.items():
        split_rows = mapped_table[mapped_table["split"] == split]
        class_rows = split_rows[split_rows["falsified"] == falsified]
        sample_count = min(sample_count, len(class_rows))
        sample_rows = class_rows.sample(n=sample_count, random_state=42)
        review_samples.append(sample_rows)

manual_review = pd.concat(review_samples, ignore_index=True)
manual_review_table = manual_review[example_columns].copy()
manual_review_table = manual_review_table.rename(columns={
    "submitted_caption_source": "caption_source",
    "query_image_source": "image_source",
})
with pd.option_context("display.max_colwidth", None, "display.max_rows", 20):
    display(manual_review_table)

,split,falsified,id,image_id,submitted_caption,query_image_original_caption,query_image_path,caption_source,image_source
0,train,False,145098,145098,A Bangladeshi man who said he was a Japanese translator wears a black ribbon on his lapel as he stands by a road block in Dhaka,A Bangladeshi man who said he was a Japanese translator wears a black ribbon on his lapel as he stands by a road block in Dhaka,visual_news/origin/washington_post/images/0566/479.jpg,washington_post,washington_post
1,train,False,85949,85949,Nevada congresswoman makes fun of Trump s short fingers,Nevada congresswoman makes fun of Trump s short fingers,visual_news/origin/washington_post/images/0345/284.jpg,washington_post,washington_post
2,train,False,193760,193760,Kerry the 2004 Democratic presidential nominee lost his bid for the White House to incumbent President George W Bush by 118000 votes in Ohio,Kerry the 2004 Democratic presidential nominee lost his bid for the White House to incumbent President George W Bush by 118000 votes in Ohio,visual_news/origin/washington_post/images/0573/638.jpg,washington_post,washington_post
3,train,False,667520,667520,Tweet thing David Bowie s Where Are We Now creates a storm on Twitter,Tweet thing David Bowie s Where Are We Now creates a storm on Twitter,visual_news/origin/guardian/images/0375/328.jpg,guardian,guardian
4,val,False,964394,964394,Indian villagers paddle through floodwaters in Araria some 220 miles from Patna on August 26,Indian villagers paddle through floodwaters in Araria some 220 miles from Patna on August 26,visual_news/origin/guardian/images/0193/792.jpg,guardian,guardian
5,val,False,323043,323043,Team USA goalie Jonathan Quick makes a save on Team Canada forward Matt Duchene,Team USA goalie Jonathan Quick makes a save on Team Canada forward Matt Duchene,visual_news/origin/usa_today/images/0249/729.jpg,usa_today,usa_today
6,val,False,1507901,1507901,Shrapnel in the Homs attack left Paul Conroy s left leg with a hole through it,Shrapnel in the Homs attack left Paul Conroy s left leg with a hole through it,visual_news/origin/bbc/images/0252/587.jpg,bbc,bbc
7,test,False,72374,72374,Hillary Clinton greets fans at a signing for her book Hard Choices at the Tattered Cover bookstore in Denver in 2014,Hillary Clinton greets fans at a signing for her book Hard Choices at the Tattered Cover bookstore in Denver in 2014,visual_news/origin/washington_post/images/0479/123.jpg,washington_post,washington_post
8,test,False,1262291,1262291,A veteran waves a flag during the parade as it passes along Whitehall in London,A veteran waves a flag during the parade as it passes along Whitehall in London,visual_news/origin/guardian/images/0087/552.jpg,guardian,guardian
9,test,False,597276,597276,David Cameron and his father Ian in April 2010,David Cameron and his father Ian in April 2010,visual_news/origin/guardian/images/0301/358.jpg,guardian,guardian


## Edge Cases

In [18]:
different_source_mask = falsified_rows["submitted_caption_source"] != falsified_rows["query_image_source"]
different_source_rows = falsified_rows[different_source_mask]
print("Falsified rows with different publishers:", len(different_source_rows))
with pd.option_context("display.max_colwidth", None):
    display(different_source_rows[example_columns].head(3))

different_id_mask = resolved_rows["id"] != resolved_rows["image_id"]
equal_caption_mask = resolved_rows["captions_equal"] == True
identical_caption_rows = resolved_rows[different_id_mask & equal_caption_mask]
print("Different IDs with identical captions:", len(identical_caption_rows))
with pd.option_context("display.max_colwidth", None):
    display(identical_caption_rows[example_columns].head(3))

Falsified rows with different publishers: 22698


,split,falsified,id,image_id,submitted_caption,query_image_original_caption,query_image_path,submitted_caption_source,query_image_source
5,train,True,1490080,1279161,Rock band Paramore is another group which sells cruises,Hot air balloons lift off on the first day of the 2015 Albuquerque International Balloon Fiesta in New Mexico,visual_news/origin/guardian/images/0089/703.jpg,bbc,guardian
7,train,True,66897,613721,Independent Afghan civil society activist women weep and lie in Farkhunda s grave,An Egyptian grave from 3400BC in the British Museum,visual_news/origin/guardian/images/0317/979.jpg,washington_post,guardian
9,train,True,217993,970480,Kansas State Troopers stand outside a police line near the location of a shooting at the Jewish Community Center in Overland Park Kan Sunday April 13 2014,The Fraternal Order of Police s president said the union had called in security contractors to investigate and the hack had been traced to an IP address in the UK,visual_news/origin/guardian/images/0195/320.jpg,usa_today,guardian


Different IDs with identical captions: 0


,split,falsified,id,image_id,submitted_caption,query_image_original_caption,query_image_path,submitted_caption_source,query_image_source


In [19]:
metadata_caption_counts = Counter()
for record in visualnews_by_id.values():
    metadata_caption_counts[record["caption"]] += 1

repeated_caption_ids = set()
for metadata_id, record in visualnews_by_id.items():
    if metadata_caption_counts[record["caption"]] > 1:
        repeated_caption_ids.add(metadata_id)

caption_duplicate_mask = mapped_table["id"].isin(repeated_caption_ids)
image_duplicate_mask = mapped_table["image_id"].isin(repeated_caption_ids)
duplicate_caption_examples = mapped_table[caption_duplicate_mask | image_duplicate_mask]
print("Mapped rows involving repeated metadata captions:", len(duplicate_caption_examples))
with pd.option_context("display.max_colwidth", None):
    display(duplicate_caption_examples[example_columns].head(3))

Mapped rows involving repeated metadata captions: 690


,split,falsified,id,image_id,submitted_caption,query_image_original_caption,query_image_path,submitted_caption_source,query_image_source
566,train,False,679905,679905,A Jacksonville Fan waves a flag at Wembley London,A Jacksonville Fan waves a flag at Wembley London,visual_news/origin/guardian/images/0390/881.jpg,guardian,guardian
567,train,True,679905,338819,A Jacksonville Fan waves a flag at Wembley London,General view of Candlestick Park during the national anthem before the NFC divisional round playoff game between the San Francisco 49ers and the Green Bay Packers,visual_news/origin/usa_today/images/0396/512.jpg,guardian,usa_today
818,train,False,208233,208233,Wreckage of the Sugar Hill RV Park is scattered following a storm in Convent La,Wreckage of the Sugar Hill RV Park is scattered following a storm in Convent La,visual_news/origin/usa_today/images/0681/131.jpg,usa_today,usa_today


In [20]:
old_years = ["1900", "1963"]
caption_years = mapped_table["caption_timestamp"].str[:4]
image_years = mapped_table["image_timestamp"].str[:4]
old_caption_mask = caption_years.isin(old_years)
old_image_mask = image_years.isin(old_years)
old_timestamp_examples = mapped_table[old_caption_mask | old_image_mask]
print("Mapped rows with metadata timestamp years 1900 or 1963:", len(old_timestamp_examples))

timestamp_columns = example_columns + ["caption_timestamp", "image_timestamp"]
with pd.option_context("display.max_colwidth", None):
    display(old_timestamp_examples[timestamp_columns].head(3))

Mapped rows with metadata timestamp years 1900 or 1963: 9


,split,falsified,id,image_id,submitted_caption,query_image_original_caption,query_image_path,submitted_caption_source,query_image_source,caption_timestamp,image_timestamp
24911,train,True,391256,44753,Clinton and Barack Obama wave to the crowd after speaking at a campaign event at the New Amsterdam Theatre on June 4 2012 in New York,Former presidents Jimmy Carter and Bill Clinton greet members of the King family,visual_news/origin/washington_post/images/0590/807.jpg,usa_today,washington_post,2016-07-27T03:16:32Z,1963-08-29T17:14:00Z
33628,train,False,44753,44753,Former presidents Jimmy Carter and Bill Clinton greet members of the King family,Former presidents Jimmy Carter and Bill Clinton greet members of the King family,visual_news/origin/washington_post/images/0590/807.jpg,washington_post,washington_post,1963-08-29T17:14:00Z,1963-08-29T17:14:00Z
33629,train,True,44753,391256,Former presidents Jimmy Carter and Bill Clinton greet members of the King family,Clinton and Barack Obama wave to the crowd after speaking at a campaign event at the New Amsterdam Theatre on June 4 2012 in New York,visual_news/origin/usa_today/images/0638/025.jpg,washington_post,usa_today,1963-08-29T17:14:00Z,2016-07-27T03:16:32Z


## Notes

- Mapping coverage checked.
- Caption-side and image-side semantics verified.
- No processed mapping file was created.
- Manual review is required before the next stage.